# Lecture 6: ODE solvers

## Schrödinger Equation (SE)

In [1]:
import sys
sys.path.append("..")

# load standard libraries
import time  # for runtime analysis
import numpy as np                  # standard numerics library
import matplotlib.pyplot as plt     # plotting library
import matplotlib.ticker as ticker

from numpy import linalg as LA      # NumPy linear algebra tools
from scipy import sparse            # sparse matrix tools
from scipy import linalg as sLA     # SciPy linear algebra tools
from scipy.sparse.linalg import eigsh  # sparse eigenvalue solver
from numpy.linalg import eigh       # Hermitian matrix diagonalization
import Comp_Quant_Dynam as cqd      # custom quantum dynamics package

%matplotlib inline

# Solve the Time-Dependent Schrödinger Equation (TDSE)

The time-dependent Schrödinger equation is

$$
\dot{\vec{\psi}}(t)
=
-iH\vec{\psi}(t),
\qquad
\vec{\psi}(0)=\vec{\psi}_0
$$

For a time-independent Hamiltonian $H$, the formal solution is

$$
\vec{\psi}(t)
=
e^{-iHt}\vec{\psi}_0
$$

We can verify that this expression satisfies the TDSE.

---

### Verification Proof

Take the time derivative:

$$
\frac{d}{dt}\vec{\psi}(t)
=
\frac{d}{dt}
\left(
e^{-iHt}\vec{\psi}_0
\right)
$$

Since $\vec{\psi}_0$ is time independent:

$$
\frac{d}{dt}\vec{\psi}(t)
=
\left(
\frac{d}{dt}e^{-iHt}
\right)\vec{\psi}_0
$$

Using the derivative rule $\frac{d}{dt}e^{-iHt} = -iHe^{-iHt}$, we obtain:

$$
\frac{d}{dt}\vec{\psi}(t)
=
-iHe^{-iHt}\vec{\psi}_0
$$

Recognizing that $e^{-iHt}\vec{\psi}_0 = \vec{\psi}(t)$, we get:

$$
\frac{d}{dt}\vec{\psi}(t)
=
-iH\vec{\psi}(t)
$$

which is exactly the time-dependent Schrödinger equation.

---

## 1. Exact Diagonalization

We can decompose the Hamiltonian as:

$$H = V D V^\dagger$$

* **Eigenvectors** of $H$ are the **columns** of $V$
* **Eigenvalues** of $H$ are on the **diagonal** of $D$

$$\Rightarrow \vec{\psi}(t) = V e^{-iDt} V^\dagger \vec{\psi}_0$$

Where $e^{-iDt}$ is the diagonal matrix:

$$
e^{-iDt} = 
\begin{pmatrix}
e^{-iE_1 t} & & 0 \\
& \ddots & \\
0 & & e^{-iE_d t}
\end{pmatrix}
$$

> **Concept:** Diagonalization means finding the eigenvalues and eigenvectors of the Hamiltonian. In quantum mechanics, we use this diagonalization not only to understand the energy spectrum of $H$, but also to compute the exact quantum time evolution of the system.

### Purpose of Exact Diagonalization
1. Diagonalize the Hamiltonian $H$.
2. Compute the exact solution of the time-dependent Schrödinger equation.

Exact diagonalization is **exact**, **stable**, and **norm conserving**. This is because the exact propagator 

$$U(t)=e^{-iHt}$$ 

is unitary and satisfies:

$$U^\dagger U = \mathbb{I}$$

---

## Exact Diagonalization Properties

| Property | Exact Diagonalization |
| :--- | :--- |
| **Exact?** | Yes |
| **Stable?** | Yes |
| **Norm conserving?** | Yes |
| **Unitary?** | Yes |
| **Error?** | None |
| **Computational cost** | Very expensive for large systems ($O(N^3)$ complexity) |

In [2]:
# Example Hamiltonian H  
H = np.array([[1, 0.2],[0.2, 2]], dtype=complex)
psi0 = np.array([1, 0], dtype=complex)
t = 1
# Exact diagonalization (H = V D V^) and Diagonal matrix D
eigenvalues, eigenvectors = eigh(H)
V = eigenvectors
D = np.diag(eigenvalues)

# psi(t) = V exp(-iDt) V^dagger psi0
# exp(-iDt) and psi(t)
expD = np.diag(np.exp(-1j * eigenvalues * t))
psi_t = V @ expD @ V.conj().T @ psi0

# Results
print("D=\n", D)
print("Eigenvalues=", eigenvalues)
print("Eigenvectors=\n", V)
print("psi(t)=", psi_t)
print("Norm=", np.vdot(psi_t, psi_t))

D=
 [[0.96148352 0.        ]
 [0.         2.03851648]]
Eigenvalues= [0.96148352 2.03851648]
Eigenvectors=
 [[-0.98195639+0.j  0.18910752+0.j]
 [ 0.18910752+0.j  0.98195639+0.j]]
psi(t)= [ 0.5357143-0.82263625j -0.1899954-0.01347349j]
Norm= (1.0000000000000004+0j)


# Limitations & Alternative Approaches

## 1. Problems with Exact Diagonalization
* **Computational Complexity:** $V$ is a **dense matrix** $\Rightarrow$ the matrix-vector multiplication $V \vec{\psi}_0$ scales as $O(d^2)$.
* **Memory Limits:** Storing dense matrices introduces severe memory limitations for large Hilbert space dimensions $d$.
* **Time-Dependence:** For problems with a time-dependent Hamiltonian $H(t)$, global exact diagonalization is completely infeasible.

> 💡 **Core Insight:** Because the equation is dynamical in time, we can use time evolution itself as the computational framework, completely avoiding expensive global diagonalization.

---

## 2. Solution: Discretize Time into Steps $\Delta t$

Instead of solving for all time at once, we break the evolution down:

$$\vec{\psi}(t) = \left(e^{-iH\Delta t}\right)^n \vec{\psi}_0 \quad \text{where } t = n \Delta t$$

$$\Rightarrow \text{The goal becomes finding: } \vec{\psi}(t+\Delta t) = e^{-iH\Delta t} \vec{\psi}(t)$$

### Comparing Analytical vs. Numerical Evolution

* **Exact Global Solution:**
  $$\psi(t) = e^{-iHt}\,\psi(0)$$

* **One-Step Evolution Solution:**
  $$\psi(t+\Delta t) = e^{-iH\Delta t}\,\psi(t)$$

* **Euler Approximation (Numerical) Solution:**
  $$\psi(t+\Delta t) \approx (1 - iH\Delta t)\,\psi(t)$$

---

## ⚠️ Very Important Concept

Think of the operators like this:

* $e^{-iHt}$ $\rightarrow$ Evolves the system all the way from **start $\rightarrow$ final time $t$**.
* $e^{-iH\Delta t}$ $\rightarrow$ Evolves the system for a **single small time step $\Delta t$ only**.

### Key Idea
$$\text{Big Time Evolution} = \text{Many Small Steps Multiplied Together}$$

---

## 3. Important Numerical Questions

For every numerical propagator we design, we must critically evaluate:

1. **Numerical Error:** What is the local and global truncation error?
2. **Stability:** Is the numerical method stable or will it blow up over time?
3. **Conservation:** Is the method norm-conserving (unitary) or does it lose/gain probability?
4. **Scaling:** How does the computational cost scale with the system size?

Because of the limitations of exact diagonalization and the Euler method, we must utilize more sophisticated techniques.

### Advanced Numerical Propagator Methods:
* **Taylor-series propagators** (e.g., Higher-order Euler methods)
* **Implicit methods** (e.g., Crank–Nicolson method)
* **Krylov-subspace propagators** (e.g., Lanczos time evolution)
* **Adaptive step-size methods** (e.g., Runge-Kutta pairs)

# 2) Taylor Series Propagators

> **Notation Note:**
> * $\Delta t \longleftrightarrow dt$
> * $\vec{\psi} \longleftrightarrow |\psi\rangle$

---

## Euler Method (1st Order Taylor)

The formal solution to the time-independent Schrödinger Equation over a small time step $dt$ is given by:

$$|\psi(t+dt)\rangle = e^{-iHdt}|\psi(t)\rangle$$

Expanding the matrix exponential as a Taylor series:

$$e^{-iHdt} = \frac{(-iHdt)^0}{0!} + \frac{(-iHdt)^1}{1!} + \frac{(-iHdt)^2}{2!} + \cdots = \mathbb{1} - iHdt + \frac{(-iHdt)^2}{2!} + \cdots$$

Since $dt$ is microscopic ($dt \ll 1$), higher-order powers become negligibly small ($(dt)^1 \gg (dt)^2 \gg (dt)^3 \dots$). Truncating after the first-order term yields the **Euler method approximation**:

$$|\psi(t+dt)\rangle \approx (\mathbb{1} - iHdt)|\psi(t)\rangle + \mathcal{O}(dt^2)$$

* **Local Error:** $\sim dt^2$ per time step.
* **Global Error:** $\sim dt$ for a fixed final time $t_{\text{end}}$.

---

### Stability Analysis: Calculating the Norm

To check if the Euler method preserves physical probability, we compute the norm of the propagated state $\langle\psi(t+dt)|\psi(t+dt)\rangle$.

First, find the bra vector using the conjugate transpose rule $(AB)^\dagger = B^\dagger A^\dagger$ and the fact that the Hamiltonian is Hermitian ($H^\dagger = H$):

$$\langle\psi(t+dt)| = \left( (\mathbb{1}-idtH)|\psi(t)\rangle \right)^\dagger = \langle\psi(t)|(\mathbb{1}+idtH)$$

Now, compute the inner product:

$$\begin{aligned}
\langle\psi(t+dt)|\psi(t+dt)\rangle &= \langle\psi(t)| (\mathbb{1}+idtH)(\mathbb{1}-idtH) |\psi(t)\rangle \\
&= \langle\psi(t)| \left( \mathbb{1} - idtH + idtH + dt^2H^2 \right) |\psi(t)\rangle \\
&= \langle\psi(t)| \left( \mathbb{1} + dt^2H^2 \right) |\psi(t)\rangle \\
&= \langle\psi(t)|\psi(t)\rangle + dt^2\langle\psi(t)|H^2|\psi(t)\rangle \\
&= 1 + dt^2\langle\psi(t)|H^2|\psi(t)\rangle
\end{aligned}$$

$$\rightarrow \text{\textbf{Norm grows!}}$$

Using the identity $(1 + \alpha)^n \approx e^{n\alpha}$, after $n_t$ total steps (where $t = n_t dt$), the cumulative norm scales as:

$$\begin{aligned}
\|\psi(t)\|^2 &= \left[ \langle\psi_0| \left( \mathbb{1} + dt^2 H^2 \right) |\psi_0\rangle \right]^{n_t} \\
&= \left[ \langle\psi_0|\psi_0\rangle + dt^2 \langle\psi_0| H^2 |\psi_0\rangle \right]^{n_t} \\
&\approx \left[ 1 + dt^2 \lambda_{\text{max}}^2 \right]^{n_t} \\
&= \left[ 1 + dt \cdot (n_t \cdot dt) \cdot \lambda_{\text{max}}^2 \right] \\
&= 1 + t \cdot dt \cdot \lambda_{\text{max}}^2 \\
&\approx e^{t \cdot dt \cdot \lambda_{\text{max}}^2}
\end{aligned}$$

*(where $\lambda_{\text{max}}$ is the largest eigenvalue of $H$)*

$$\Rightarrow \text{\textbf{Norm explodes exponentially}} \Rightarrow \text{\textbf{Violently Unstable!}}$$

$$\Rightarrow \text{Requires unphysically tiny time steps: } dt \ll \frac{1}{\lambda_{\text{max}}}$$

---

## $n$-th Order Taylor Propagators

Instead of stopping at first order, we can keep more terms of the expansion:

$$|\psi(t+dt)\rangle = \left( \sum_{j=0}^{n} \frac{(-idt)^j}{j!} H^j \right) |\psi(t)\rangle + \mathcal{O}(dt^{n+1})$$

* **Error per step:** $\sim dt^{n+1}$, which allows for larger time steps than Euler.
* **Stability:** **Still unstable**. Regardless of the truncation order $n$, a polynomial expansion of an imaginary exponential is never strictly unitary, meaning the norm will eventually drift or explode.

> **Note:** For any linear differential equation of the form $\dot{\psi} = H\psi$, standard explicit Runge-Kutta methods (like RK4) simplify exactly to this truncated Taylor expansion method.

---

## General Stability & System Constraints

### General Stability Criterion
For any generic propagation step $|\psi(t+\Delta t)\rangle = A|\psi(t)\rangle$:

$$\text{If } \|A^\dagger A\|_2 > 1 \longrightarrow \text{\textbf{unstable}}$$

*(where $\|\cdot\|_2$ is the matrix 2-norm, corresponding to the absolute value of its largest eigenvalue)*

### Stiffness
A quantum mechanical problem is considered **stiff** when:

$$\left| \frac{\lambda_{\text{max}}}{\lambda_{\text{min}}} \right| \gg 1$$

* **Physical Reality:** You must pick an incredibly small $dt$ to resolve the rapid dynamics dictated by the largest eigenvalue ($1/\lambda_{\text{max}}$). However, to observe the slow overall system behavior dictated by the smallest eigenvalue, you must simulate for a long duration ($1/\lambda_{\text{min}}$). 
$$\Rightarrow \text{\textbf{Massive number of total simulation steps needed.}}$$

### Advantage of Explicit Methods
Despite stability limitations, explicit Taylor/Euler steps have one massive computational benefit:
* They only require **matrix-vector multiplication** ($H|\psi\rangle$).
$$\rightarrow \text{\textbf{Highly efficient if }} H \text{\textbf{ is sparse, scaling linearly as }} \mathcal{O}(d).$$

---

## Summary

* The Euler method is **not norm-conserving** because its propagator matrix $(\mathbb{1} - iHdt)$ is **non-unitary**.
* As a result, the wavefunction norm grows exponentially over time, causing numerical explosion.
* Increasing the polynomial order ($n$-th order Taylor or Runge-Kutta) reduces local error but does not fix the underlying non-unitarity. 
* Consequently, stable long-term quantum propagation strictly demands **norm-conserving (unitary) methods** (e.g., Crank-Nicolson or Krylov Subspace methods).

# 3) Implicit Methods: Crank–Nicolson

The Euler method is inherently unstable for quantum systems because its propagator is non-unitary, meaning it fails to conserve the norm of the wavefunction. To fix this, we look toward numerical propagation schemes that are stable and strictly norm-conserving during quantum time evolution. 

The **Crank–Nicolson method** achieves this stability by taking an **implicit** approach. Instead of evaluating the system's dynamics purely at the current time $t$ (explicit), it balances information from both the current state at time $t$ and the future state at time $t + dt$.

Instead of using only the current time step, it uses information from both:

. the current state (t)
. the next state (t+dt)

## The Core Idea & Derivation

We begin with the formal time-evolution step over an interval $\Delta t$ (or $dt$):

$$|\psi(t+\Delta t)\rangle = e^{-iH\Delta t} |\psi(t)\rangle$$

We can symmetrically split the exponential propagator into two half-steps:

$$e^{-iH\Delta t} = e^{-iH\Delta t/2} e^{-iH\Delta t/2}$$

Substituting this back into the evolution equation gives:

$$|\psi(t+\Delta t)\rangle = e^{-iH\Delta t/2} e^{-iH\Delta t/2} |\psi(t)\rangle$$

Multiplying both sides from the left by $e^{+iH\Delta t/2}$ structurally isolates the future state and current state on opposite sides:

$$e^{+iH\Delta t/2} |\psi(t+\Delta t)\rangle = e^{-iH\Delta t/2} |\psi(t)\rangle$$

Next, we apply a first-order Taylor expansion to both sides for their respective half-steps:

$$e^{+iH\Delta t/2} \approx \mathbb{1} + i \frac{\Delta t}{2} H$$

$$e^{-iH\Delta t/2} \approx \mathbb{1} - i \frac{\Delta t}{2} H$$

Substituting these approximations back results in the foundational algebraic relation for the Crank–Nicolson scheme:

$$\left( \mathbb{1} + i \frac{\Delta t}{2} H \right) |\psi(t+\Delta t)\rangle = \left( \mathbb{1} - i \frac{\Delta t}{2} H \right) |\psi(t)\rangle$$

---

## Linear System Representation ($Ax = b$)

To see why this method is structurally "implicit," let $\Delta t = dt$ and define the system elements as a standard linear algebra problem:

$$\underbrace{\left(\mathbb{1} + i \frac{dt}{2} H\right)}_{A} \underbrace{|\psi(t+dt)\rangle}_{x} = \underbrace{\left(\mathbb{1} - i \frac{dt}{2} H\right) |\psi(t)\rangle}_{b}$$

* **Why it is called "Implicit":** To solve for the future state vector $x = |\psi(t+dt)\rangle$, we cannot simply compute a direct matrix multiplication. Instead, we must solve a full linear system of equations $Ax = b$ at every single time step.
* **Computational Trade-off:** Solving a linear system is computationally more expensive per step than explicit methods. However, because it is unconditionally stable, **we can use significantly larger time steps ($dt$)**, which often makes it much more efficient overall.

Formally, the solution is written by applying the matrix inverse:

$$|\psi(t+dt)\rangle = \underbrace{\left( \mathbb{1} + i \frac{dt}{2} H \right)^{-1} \left( \mathbb{1} - i \frac{dt}{2} H \right)}_{A} |\psi(t)\rangle$$

---

## Key Mathematical Properties

### 1. Unitary & Norm-Conserving
Let our effective Crank–Nicolson propagator be defined as $A = \frac{\mathbb{1} - i \frac{dt}{2} H}{\mathbb{1} + i \frac{dt}{2} H}$. Let's evaluate its absolute stability by checking $A^\dagger A$:

$$A^\dagger A = \left( \frac{\mathbb{1} - i \frac{dt}{2} H}{\mathbb{1} + i \frac{dt}{2} H} \right)^\dagger \left( \frac{\mathbb{1} - i \frac{dt}{2} H}{\mathbb{1} + i \frac{dt}{2} H} \right) = \left( \frac{\mathbb{1} + i \frac{dt}{2} H}{\mathbb{1} - i \frac{dt}{2} H} \right) \left( \frac{\mathbb{1} - i \frac{dt}{2} H}{\mathbb{1} + i \frac{dt}{2} H} \right) = \mathbb{1}$$

$$A^\dagger A = \mathbb{1} \Rightarrow A \text{ is strictly unitary.}$$

Because the operator is unitary, the norm of the state vector is conserved perfectly over time ($\|A\|_2 = 1$). The total physical probability remains exactly $1$, meaning **the method cannot blow up or explode.**

### 2. Truncation Error
* **Local Error:** $\sim dt^3$ per step.
* Because the split expansion balances terms symmetrically around the midpoint, the second-order error terms cancel out completely, making it a highly accurate second-order global method ($\mathcal{O}(dt^2)$ overall).

# Crank–Nicolson: Formal Local & Global Error Derivation

This notebook provides a rigorous step-by-step mathematical derivation of the truncation error for the Crank–Nicolson time-evolution operator compared against the exact exponential solution of the time-dependent Schrödinger Equation.

---

## 1. Exact Time Evolution (The Reference Target)

For a time-independent Hamiltonian $H$, the Schrödinger Equation is defined as:

$$i \frac{d}{dt}|\psi(t)\rangle = H|\psi(t)\rangle$$

Over a discrete forward time step $\Delta t$, the analytical solution is driven by the exact unitary exponential propagator:

$$|\psi_{\text{exact}}(t+\Delta t)\rangle = e^{-iH\Delta t}|\psi(t)\rangle$$

Expanding this matrix exponential using its foundational Taylor series yields:

$$e^{-iH\Delta t} = \mathbb{1} - iH\Delta t - \frac{1}{2}H^2\Delta t^2 + \mathcal{O}(\Delta t^3)$$

Therefore, our target reference state vector is:

$$|\psi_{\text{exact}}(t+\Delta t)\rangle = \left(\mathbb{1} - iH\Delta t - \frac{1}{2}H^2\Delta t^2 + \mathcal{O}(\Delta t^3)\right)|\psi(t)\rangle \quad \text{--- (Eq. 1)}$$

---

## 2. The Crank–Nicolson Approximation

The Crank–Nicolson approximation models this system implicitly using an effective discrete operator $U_{\text{CN}}$:

$$|\psi_{\text{CN}}(t+\Delta t)\rangle = U_{\text{CN}} |\psi(t)\rangle = A^{-1}B |\psi(t)\rangle$$

Where the component operators $A$ and $B$ are defined via half-step parameterizations:

$$A = \mathbb{1} + \frac{iH\Delta t}{2}, \quad B = \mathbb{1} - \frac{iH\Delta t}{2}$$

This gives the full fraction-style operator representation:

$$U_{\text{CN}} = \left(\mathbb{1} + \frac{iH\Delta t}{2}\right)^{-1} \left(\mathbb{1} - \frac{iH\Delta t}{2}\right)$$

---

## 3. Taylor Expansion of the Crank–Nicolson Operator

To compare $U_{\text{CN}}$ directly against the exact exponential propagator, we evaluate its parts using a matrix power series expansion.

### Step 1: Inverse Operator Expansion
Using the generic geometric matrix series identity $(\mathbb{1} + X)^{-1} = \mathbb{1} - X + X^2 - X^3 + \cdots$, let $X = \frac{iH\Delta t}{2}$:

$$\left(\mathbb{1} + \frac{iH\Delta t}{2}\right)^{-1} = \mathbb{1} - \left(\frac{iH\Delta t}{2}\right) + \left(\frac{iH\Delta t}{2}\right)^2 - \mathcal{O}(\Delta t^3)$$

Evaluating the imaginary powers ($i^2 = -1$):

$$\left(\mathbb{1} + \frac{iH\Delta t}{2}\right)^{-1} = \mathbb{1} - \frac{iH\Delta t}{2} - \frac{H^2\Delta t^2}{4} + \mathcal{O}(\Delta t^3)$$

### Step 2: Multiplication with Operator $B$
Now, multiply this inverse series expansion directly onto operator $B$:

$$U_{\text{CN}} = \left[ \mathbb{1} - \frac{iH\Delta t}{2} - \frac{H^2\Delta t^2}{4} + \mathcal{O}(\Delta t^3) \right] \left[ \mathbb{1} - \frac{iH\Delta t}{2} \right]$$

Distributing the multiplication terms out systematically across their corresponding parameter orders:

* **$\mathcal{O}(1)$ Term:** $$\mathbb{1} \cdot \mathbb{1} = \mathbb{1}$$
* **$\mathcal{O}(\Delta t)$ Terms:** $$\left(\mathbb{1} \cdot -\frac{iH\Delta t}{2}\right) + \left(-\frac{iH\Delta t}{2} \cdot \mathbb{1}\right) = -iH\Delta t$$
* **$\mathcal{O}(\Delta t^2)$ Terms:** $$\left(-\frac{iH\Delta t}{2} \cdot -\frac{iH\Delta t}{2}\right) + \left(-\frac{H^2\Delta t^2}{4} \cdot \mathbb{1}\right) = \left(\frac{i^2 H^2 \Delta t^2}{4}\right) - \frac{H^2\Delta t^2}{4} = -\frac{H^2\Delta t^2}{4} - \frac{H^2\Delta t^2}{4} = -\frac{1}{2}H^2\Delta t^2$$

Combining these sorted order elements reveals the total approximated profile:

$$U_{\text{CN}} = \mathbb{1} - iH\Delta t - \frac{1}{2}H^2\Delta t^2 + \mathcal{O}(\Delta t^3) \quad \text{--- (Eq. 2)}$$

---

## 4. Comparing the Approximations

Let us match our expanded expressions back together:

$$\text{Exact Exponential (Eq. 1): } \quad \mathbb{1} - iH\Delta t - \frac{1}{2}H^2\Delta t^2 + \mathcal{O}(\Delta t^3)$$
$$\text{Crank–Nicolson (Eq. 2): } \quad \mathbb{1} - iH\Delta t - \frac{1}{2}H^2\Delta t^2 + \mathcal{O}(\Delta t^3)$$

The algebraic profiles **match completely up to second-order terms ($\Delta t^2$)**. 

---

## 5. Truncation Error Analysis

### Local Truncation Error (Per Step)
The Local Truncation Error (LTE) is the variance introduced across a single isolated discrete leap:

$$\text{LTE} = U_{\text{CN}} - e^{-iH\Delta t} = \mathcal{O}(\Delta t^3)$$

$$\boxed{\text{Local Error Per Step} = \mathcal{O}(\Delta t^3)}$$

### Global Truncation Error (Accumulated)
The Global Truncation Error (GTE) tracks total error accumulated over a macro timeline $T$. The total number of steps required is $N = \frac{T}{\Delta t}$. 

Multiplying the single-step error across the full structural runtime path yields:

$$\text{GTE} = N \cdot \text{LTE} = \left(\frac{T}{\Delta t}\right) \cdot \mathcal{O}(\Delta t^3) = T \cdot \mathcal{O}(\Delta t^2)$$

$$\boxed{\text{Global Accumulated Error} = \mathcal{O}(\Delta t^2)}$$

---

## Summary Summary

* **Crank–Nicolson** serves as an implicit approximation of the exact quantum exponential propagator.
* By using a split-step balance approach, it matches the true Taylor series scaling exactly up to $\Delta t^2$.
* This sets a local error of $\mathcal{O}(\Delta t^3)$, rendering the method an **unconditionally stable, 2nd-order globally accurate ($\mathcal{O}(\Delta t^2)$)** quantum simulator.

## 4) Krylov subspace propagator

Recall $n$-th order Taylor series:

$$|\psi(t+dt)\rangle = \left( 1 - idtH - \frac{1}{2} dt^2 H^2 \pm \dots \right) |\psi(t)\rangle$$

$$= \text{superpos. of vectors } (-iH)^k |\psi(t)\rangle \quad (k = 0 \dots n)$$
$$\text{with decreasing weight } \frac{dt^k}{k!}$$

$$\rightarrow |\psi(t+dt)\rangle \text{ lies in Krylov space } \mathcal{K}_{n+1}(-iH, |\psi(t)\rangle)$$
$$\text{spanned by vectors } (-iH)^k |\psi(t)\rangle \text{ up to an error } \sim dt^{n+1}$$

---

### Idea: Evolve unitarily in Krylov space.

Recall: $\{q_0, \dots, q_n\}$ are orthonormalized Krylov vectors.

$$Q = \begin{pmatrix} q_0 \dots q_n \end{pmatrix} \updownarrow d \quad \text{rectangular matrix} \quad (\leftrightarrow n)$$

$$h = Q^\dagger H Q \quad \text{Hamiltonian within Krylov space}$$

---

### "Algorithm":
* Project $|\psi(t)\rangle$ into Krylov space
* Evolve with $e^{-ihdt}$. Can be done exactly as $h$ ($n \times n$ tri-diagonal matrix is easily diagonalized)
* Express $|\psi(t+dt)\rangle$ in original basis

---

### Mathematically:

$$|\psi(t+dt)\rangle = Q e^{-ihdt} Q^\dagger |\psi(t)\rangle$$

$$= Q e^{-ihdt} e_1$$

$$\text{(recall that } |\psi(t)\rangle \text{ is the first Krylov-vector } q_0)$$

$$\rightarrow Q^\dagger |\psi(t)\rangle = \begin{pmatrix} 1 \\ 0 \\ \vdots \\ 0 \end{pmatrix} \updownarrow n \equiv e_1 \text{ )}$$

This scheme is obviously unitary (norm-preserving) since $Q e^{-ihdt} Q^\dagger$ is a unitary operator.

**Claim:** This scheme is an order $dt^{n+1}$ propagation just as the $n$-th order Taylor series propagator.

$$Q e^{-ihdt} Q^\dagger = e^{-i Q h Q^\dagger dt} \equiv e^{-i \tilde{H} dt} \overset{!}{=} e^{-i H dt} + \mathcal{O}(dt^{n+1})$$

To prove the last equality, we need to show that:

$$H^k \psi = \tilde{H}^k \psi \quad \text{for } k \le n$$

---

### Proof by induction:

Assuming $H^{k-1} \psi = \tilde{H}^{k-1} \psi$, we have:

$$H^k \psi = \underbrace{Q Q^\dagger H^k \psi}_{H^k \psi \text{ is inside of } \mathcal{K}_{n+1}} = Q Q^\dagger H \underbrace{\tilde{H}^{k-1} \psi}_{\text{ind. assumption}} = Q Q^\dagger H Q h^{k-1} Q^\dagger \psi$$

$$= Q h^k Q^\dagger \psi = \tilde{H}^k \psi$$

$$\begin{pmatrix} q_0 \dots q_n \end{pmatrix} \begin{pmatrix} q_0^\dagger \\ \vdots \\ q_n^\dagger \end{pmatrix} \underbrace{\sum_{k=0}^n c_k q_k}_{\parallel \atop H^k \psi} = \begin{pmatrix} q_0 \dots q_n \end{pmatrix} \begin{pmatrix} c_0 \\ \vdots \\ c_n \end{pmatrix} = \sum c_k q_k = H^k \psi$$


### Advantages:

* ✅ **Higher Precision:** The truncation error is strictly smaller than a standard $n$-th order Taylor expansion.
* ✅ **Strictly Unitary:** Explicitly norm-preserving over long simulations.
* ✅ **Matrix-Free:** Requires only matrix-vector multiplications ($H|\psi\rangle$). There is absolutely no need for costly matrix inversions or global diagonalization.
* ✅ **Efficiency:** Typically, a subspace size of $n \sim 20$ is optimal, enabling significantly larger time steps $\Delta t$.
* Error strictly smaller than for $n$-th order Taylor.
* Norm-preserving.
* Only $H\psi$-multiplications (no matrix inversion).

  
Typically $n \sim 20$ is optimal $\Rightarrow$ large time steps possible.

**"Gold standard"** for propagating SE with sparse $H$.
* Error estimates available.

## 5) Adaptive step-size

**Goal:** In each time step, make the step size $dt$ small enough to meet a given accuracy $\varepsilon$.

* If an error estimate is available (upper bound on error), check and reduce step size by some factor until $\Delta_{\text{err}} \le \varepsilon$. If $\Delta_{\text{err}} < \varepsilon$, keep or increase $dt$.

* **Error estimation by half step:**
  * Calculate $\psi(t+dt)$ with step $dt$
  * Calculate $\tilde{\psi}(t+dt)$ by evolving in 2 half steps $\frac{dt}{2}$
  
$$\rightarrow \text{error estimate } \Delta_{\text{err}} = \|\psi(t+dt) - \tilde{\psi}(t+dt)\|$$

| Method         | Cheap?         | Sparse-friendly? | Accurate? | Stable?           | Norm conserving? |
| -------------- | -------------- | ---------------- | --------- | ----------------- | ---------------- |
| Euler          | Yes            | Yes              | Poor      | No                | No               |
| Higher Taylor  | Moderate       | Yes              | Better    | Still problematic | No               |
| Crank–Nicolson | More expensive | Yes              | Good      | Yes               | Yes              |
| Lanczos/Krylov | Excellent      | Excellent        | Excellent | Usually           | Nearly/unitary   |
